In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [2]:

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import einops
import numpy as np
import pandas as pd
import sympy as sp
import random
import scipy.stats
from scipy.stats import poisson
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchsummary import summary
import torch.optim as optim
import os
import json
import datetime
from google.colab import drive
from enum import Enum
import itertools
import shutil
import subprocess
import math


In [3]:
#Parameter-Sweep für Automatisierung vorbereiten

seed_and_error_type = {
    "random_seed": [5, 8, 13, 20, 39],
    "error_type": [
        "no_error", "double_poisson", "uniform", "identic",
        "3_summands", "randomly_dist", "cust_mu_sigma", "simple_deviation",
        "label_shuffle", "skellam", "vonmises", "special"],
}

double_poisson_params = {
   "double_poisson_lambdaY": [0.001, 0.01, 0.1, 0.2, 0.5, 1.0],
   "double_poisson_lambdaZ": [0.001, 0.01, 0.1, 0.2, 0.5, 1.0],
   "double_poisson_p": [0.3, 0.5, 0.7],
   "noise_type": ["a", "b", ""],
}

uniform_params = {
    "uniform_k": [1, 3, 5, 7, 9],
    "noise_type": ["a", "b", ""],
}

identic_params = {
    "noise_type": ["a", "b", ""],
}

summands3_params = {
    "summands3_primes": [11, 13, 17, 19],
    "summands3_weight_decay": [1.0, 2.0, 5.0, 10.0],
    "summands3_train_data_fraction": [0.1, 0.2, 0.3, 0.4]
}

randomly_dist_params = {
    "randomly_dist_rangefactor": [2, 5, 8, 15],
    "noise_type": ["a", "b", ""],
}

cust_mu_sigma_params = {
    "cust_mu_sigma_targetmean": [0.0, 1.0, 2.0],
    "cust_mu_sigma_targetvariance": [0.005, 0.01, 0.1, 1.0, 5.0],
    "cust_mu_sigma_rangefactor": [0.2, 0.5, 0.8],
    "noise_type": ["a", "b", ""],
}

simple_deviation_params = {
    "simple_deviations_prob_zero": [0.5, 0.9, 0.95, 0.99, 0.995, 0.999],
    "noise_type": ["a", "b", ""],
}

label_shuffle_params = {
    "noise_type": ["a", "b", ""],
}

skellam_params = {
    "skellam_lambda1": [0.01, 0.1, 1.0, 2.0],
    "skellam_lambda2": [0.01, 0.1, 1.0, 2.0],
    "noise_type": ["a", "b", ""],
}

vonmises_params = {
    "mu_lin": [-3, -1, 0, 1, 5, 20],
    "kappa": [0.1, 1.0, 10, 100, 500, 1000, 5000, 10000],
    "noise_type": ["a", "b", ""],
}


special_params = {
    "special_n_epochs": [30000, 60000, 100000],
    "special_model_type": ["1_hidden", "2_hidden", "3_hidden"],
    "special_weight_tying": [True, False],
    "special_d_model": [16, 32, 64, 128],
    "special_learning_rate": [0.00001, 0.0001, 0.001, 0.1],
    "special_weight_decay": [0.0, 0.1, 0.5, 1.0, 2.0],
    "special weight_noise": [True, False],
    "special_train_data_fraction": [0.1, 0.2, 0.3, 0.4]
}



def parameter_sweep(error_type = None):
    result = []
    seed_and_type = list(seed_and_error_type.keys())
    seed_and_type_values = list(seed_and_error_type.values())
    seed_and_type_combinations = list(itertools.product(*seed_and_type_values))
    if error_type:
        seed_and_type_combinations = [t for t in seed_and_type_combinations if t[1] == error_type]

    for t in seed_and_type_combinations:
        match t[1]:
            case "no_error": error_type_params = {}
            case "double_poisson": error_type_params = double_poisson_params
            case "uniform": error_type_params = uniform_params
            case "identic": error_type_params = identic_params
            case "3_summands": error_type_params = summands3_params
            case "randomly_dist": error_type_params = randomly_dist_params
            case "cust_mu_sigma": error_type_params = cust_mu_sigma_params
            case "simple_deviation": error_type_params = simple_deviation_params
            case "label_shuffle": error_type_params = label_shuffle_params
            case "skellam": error_type_params = skellam_params
            case "vonmises": error_type_params = vonmises_params
            case "special": error_type_params = special_params
            case _: error_type_params = {}

        error_type_params_values = list(error_type_params.values())

        result += [t + s for s in list(itertools.product(*error_type_params_values))]

    return result


In [4]:
os.environ["TORCH_DISABLE_DYNAMO"] = "1"
os.makedirs("/kaggle/working/results", exist_ok=True)
os.makedirs("/kaggle/working/results/runs", exist_ok=True)
os.makedirs("/kaggle/working/results/general", exist_ok=True)
os.makedirs("/kaggle/working/support", exist_ok=True)
help_path = "/kaggle/working/testing"
os.makedirs(help_path, exist_ok=True)

#Input-Handling:
input_path="/kaggle/input/database3/support"
inputIndexPath = os.path.join(input_path, "parameterIndex.txt")
inputListPath = os.path.join(input_path, "parameterList.txt")
inputForbiddenIndicesPath = os.path.join(input_path, "forbiddenIndices.txt")
parameterListPath = "/kaggle/working/support/parameterList.txt"
parameterIndexPath = "/kaggle/working/support/parameterIndex.txt"
forbiddenIndicesPath = "/kaggle/working/support/forbiddenIndices.txt"


if os.path.exists(inputIndexPath):
    with open(inputIndexPath, "r") as file:
        index = file.read()
    with open(parameterIndexPath, "w") as file:
        file.write(index)
        
if os.path.exists(inputListPath):
    with open(inputListPath, "r") as file:
        listContent = file.read()
    with open(parameterListPath, "w") as file:
        file.write(str(listContent))

if os.path.exists(inputForbiddenIndicesPath):
    with open(inputForbiddenIndicesPath, "r") as file:
        listContent2 = file.read()
    with open(forbiddenIndicesPath, "w") as file:
        print(listContent2)
        file.write(str(listContent2))
    

#Aktueller-Index-Datei erzeugen, falls noch nicht vorhanden
if not os.path.exists(inputIndexPath):
    with open(parameterIndexPath, "w") as file:
        file.write("-1")


#Datei mit Liste aller aktuellen Parameter erzeugen, falls noch nicht vorhanden
if not os.path.exists(inputListPath):
  with open (parameterListPath, "w") as file:
      s = ""
      param_sweep = parameter_sweep()
      for i in range(len(param_sweep)):
          s += str(param_sweep[i]) + "\n"
      file.write(s)

#Verbotene Indices Datei erzeugen, falls nicht existent
if not os.path.exists(inputForbiddenIndicesPath):
    with open(forbiddenIndicesPath, "w") as file:
        file.write("")
        

print(torch.cuda.is_available())
#print(torch.accelerator.current_accelerator().type)
device = ("cuda" if torch.cuda.is_available() else "cpu")

True


In [5]:

#zufällige Primzahl
def generate_prime(low_bound, up_bound):
    lowP=low_bound
    upP=up_bound
    primes = list(sp.primerange(lowP, upP))
    P = random.choice(primes)
    print(P)
    return P


#Datensatz der Addition mod P für eine Primzahl P
def get_modulo_data(P):
    data = []
    for a in range(0, P): #### (1,P) bei Modulo-Addition
        for b in range(0, P): ###(1,P) bei Modulo-Addition
            c = (a+b)%P ####(a*b) % P bei Modulo-Addition
            data.append([a, b, c])
    data = pd.DataFrame(data=data, columns= ["a", "b", "c"])
    data.index.name = "Index"
    #print(data)
    #print(data.shape)

    return data


#Fehlermaß Mean Circular Error, zusätzliches Maß neben Varianz
def get_mean_circular_error(prime, x_vec):

  errors = [min(int(np.abs(k)), prime - int(np.abs(k))) for k in x_vec]

  return np.mean(errors)


#Metriken Accuracy, Precision, Recall und F1-Score erzeugen
def get_metrics(logits_2, y_2, count, signal=None):
  if len(logits_2.shape) == 3:
          logits_2 = logits_2[:, -1]
  logits_2 = logits_2.to(torch.float64)

  logits_2_softmax = logits_2.softmax(dim=-1)
  y_pred_2 = torch.argmax(logits_2_softmax, dim=-1) #; print(y_pred_2)
  y_2 = y_2.detach().cpu().numpy().astype(np.int64)
  y_pred_2 = y_pred_2.detach().cpu().numpy().astype(np.int64)

  acc = accuracy_score(y_2, y_pred_2)
  prec = precision_score(y_2, y_pred_2, average="macro", zero_division=0.0)
  rec = recall_score(y_2, y_pred_2, average="macro", zero_division=0.0)
  f1 = f1_score(y_2, y_pred_2, average="macro")

  if count % 10000 == 0:
    print(f"---------iteration {count}----------")
    print(f"Accuracy: {acc}")
    print(f"Precision: {prec}")
    print(f"Recall: {rec}")
    print(f"F1-Score: {f1}")

  if signal == "pred":
    print(y_pred_2.shape)
    print(y_2.shape)
    data = np.array([y_pred_2, y_2.squeeze()]).T
    df = pd.DataFrame(data=data, columns=["model predictions", "true values"])
    print(df)
    rows_with_pred_error = df[df["model predictions"] != df["true values"]]
    print(f"Es wurden {len(rows_with_pred_error)} falsche Vorhersagen festgestellt (insgesamt {len(df)} Vorhersagen):")
    print(rows_with_pred_error)

    return len(rows_with_pred_error), len(df)

  return [acc, prec, rec, f1]



  #Methode, um einen effizienten Train- Test- Split durchzuführen

def get_train_test_split_results(prime=101, threeSummands=False, validationDataRequired=False, train_data_fraction=0.3):

    train_frac=train_data_fraction

    if threeSummands:
        dataset = get_modulo_data_3_summands(prime)
        X = dataset[["a", "b", "c"]]
        y = dataset [[ "d" ]]
        print(f"Test in get_train_test_split_results{X.head}")

    else:
        dataset = get_modulo_data(prime)
        X = dataset[["a", "b"]]
        y = dataset [[ "c" ]]

    X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=train_frac, random_state=8)

    if not validationDataRequired:
        return [X_train, X_test, y_train, y_test]

  #Falls neben Trainings- und Testdaten noch ein Teil der Daten (dem Modell unbekannte) Validierungsdaten bilden sollen:
    else:
        X_test_2, X_val, y_test_2, y_val = train_test_split(X_test, y_test, train_size=0.9, random_state=8)
        return [X_train, X_test_2, y_train, y_test_2, X_val, y_val]



#Fehlerterm-Modellierung:


#Idee 1: "doppelter Poisson"

#Definition "doppelter Poisson"
def double_poisson(x, lambdaY, lambdaZ, p):

    if  x >= 0:
        return p*poisson.pmf(x, mu=lambdaY)

    else:
        return (1-p)*poisson.pmf(-x, mu=lambdaZ)

#Funktion, um die die Daten inklusive Fehlerterm nach Idee 1 zu erhalten
def get_double_poisson_mod_data(prime=101, lambdaY=None, lambdaZ=None, p=None, validationDataRequired=False, inputError=""):

    if not lambdaY:
        lambdaY = 0.2 * (prime-1)
    if not lambdaZ:
        lambdaZ = 0.2 * (prime-1)
    if not p:
        p=0.5

    probabilities = {}

    for i in range(0, prime):
        xmax = min(i, prime-1-i)
        for j in range(-xmax, xmax+1):
            probabilities[j] = double_poisson(j, lambdaY, lambdaZ, p)

    probs_sorted = dict(sorted(probabilities.items()))

    int_keys = np.array([int(k) for k in probs_sorted.keys()])
    float_probs = np.array([float(l) for l in probs_sorted.values()])
    float_probs_norm = float_probs/float_probs.sum()

    tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)

    x_vec = np.random.choice(int_keys, size = len(tts_results[2]), p=float_probs_norm)

    variance = np.var(x_vec)
    mean_circular_error = get_mean_circular_error(prime, x_vec)

    if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec.astype(int).reshape(-1)) % prime

    else:

      tts_results[2]["c"] = (tts_results[2]["c"].to_numpy(dtype=int) + x_vec.astype(int).reshape(-1)) % prime

    return tts_results, variance, mean_circular_error



#Idee 2: (Symmetrische) Diskrete Gleichverteilung

def get_uniform_mod_data(prime=101, k=7, validationDataRequired=False, inputError=""):

    tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)
    values = []
    probs = []
    lower = int(-0.5*(k-1))
    upper = int(0.5*(k+1))
    for i in range(lower, upper, 1):
        values.append(i)
        probs.append(1/k)

    x_vec_2 = np.random.choice(values, size=len(tts_results[2]), p=probs)
    variance = np.var(x_vec_2)
    mean_circular_error = get_mean_circular_error(prime, x_vec_2)

    if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec_2.astype(int).reshape(-1)) % prime

    else:

      tts_results[2]["c"]= (tts_results[2]["c"].to_numpy(dtype=int) + np.array(x_vec_2).reshape(-1)) % prime

    return tts_results, variance, mean_circular_error



#Idee 3: Fehler zufällig, aber konstant/gleich für alle Inputs

def get_identic_error_mod_data(prime=101, lower_bound=None, upper_bound=None, validationDataRequired=False, inputError=""):

    lower_bound = -prime+1 #Grenze einstellen zB -3
    upper_bound = prime-1 #Grenze einstellen  zB 3
    u = random.randint(lower_bound, upper_bound)

    tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)

    if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + u) % prime

    else:

      tts_results[2]["c"]= (tts_results[2]["c"].to_numpy(dtype=int) + u) % prime

    variance = 0.0
    mean_circular_error = get_mean_circular_error(prime, np.repeat(u, prime))

    return tts_results, variance, mean_circular_error


#Idee 4: Fehler gemäß einer gänzlich zufälligen Wahrscheinlichkeitsverteilung (NICHT  zentriert, Erwartungswert nicht unbedingt 0)
#Anmerkung: über range_of_values und center lässt sich ein Intervall ganzer Zahlen für die möglichen Abweichungen spezifizieren

def get_randomly_distributed_error_mod_data(prime=101, range_of_values=None, center=0, validationDataRequired=False, inputError=""):

    if range_of_values:
        rnge = range_of_values
    else:
        rnge = int(0.2*prime)

    if rnge % 2 == 0:
        possible_values = [k for k in range(int(-0.5*rnge+1) + center, int(0.5*rnge+1) + center)]
    else:
        possible_values = [k for k in range(int(-0.5*rnge+0.5) + center, int(0.5*rnge+0.5) + center)]


    probs3 = np.random.uniform(0,1, size=rnge)
    probs3_norm = probs3 / probs3.sum()

    tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)

    x_vec_4 = np.random.choice(possible_values, size=len(tts_results[2]), p=probs3_norm)

    variance = np.var(x_vec_4)
    mean_circular_error = get_mean_circular_error(prime, x_vec_4)

    if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec_4.astype(int).reshape(-1)) % prime

    else:

      tts_results[2]["c"] = (tts_results[2]["c"].to_numpy(dtype=int) + x_vec_4.astype(int).reshape(-1)) % prime

    return tts_results, variance, mean_circular_error


#Idee 5: Erwartungswert und Varianz der Abweichung frei einstellen (Spannweite + Zentrum wieder einstellbar)

#Bestimme hier eine Wahrscheinlichkeitsverteilung mittels scipy.optimize

def get_customized_mu_sigma_modulo_data_no_2 (prime = 101, range_of_values = None, center = 0, target_mean = 0, target_var = 1, validationDataRequired=False, inputError=""):

    if range_of_values:
        rnge = range_of_values
    else:
        rnge = int(0.2*prime)

    if rnge % 2 == 0:
        possible_values = [k for k in range(int(-0.5*rnge+1) + center, int(0.5*rnge+1) + center)]
    else:
        possible_values = [k for k in range(int(-0.5*rnge+0.5) + center, int(0.5*rnge+0.5) + center)]

    x = np.array(possible_values, dtype=float)
    n = len(x)
    if n < 3:
        print("At least 3 different values are required for X")

    def objective(p):
        return np.sum(p**2)
    def normation_constraint(p):
        return np.sum(p) - 1
    def mean_constraint(p):
        return np.sum(p*x) - target_mean
    def var_constraint(p):
        return np.sum(p*(x - target_mean)**2) - target_var

    #Nebenbedingungen als Funktionen
    constraints = [
        {"type": "eq", "fun": normation_constraint},
        {"type": "eq", "fun": mean_constraint},
        {"type": "eq", "fun": var_constraint}
    ]

    #Boundaries / NNB
    bounds = [(0,1) for i in range(n)]

    #Startwerte / initial values
    p0 = np.ones(n) / n

    optimization = scipy.optimize.minimize(objective, p0, method = "SLSQP", bounds=bounds, constraints=constraints)

    if optimization.success:
        p_opt = optimization.x
        print("Wahrscheinlichkeiten:", p_opt)
        print("Summe:", np.sum(p_opt))
        print("Erwartungswert:", np.sum(p_opt*x))
        print("Varianz:", np.sum(p_opt*(x-np.sum(p_opt*x))**2))

    tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)

    x_vec_5 = np.random.choice(possible_values, size = len(tts_results[2]), p=p_opt)

    variance = np.var(x_vec_5)
    mean_circular_error = get_mean_circular_error(prime, x_vec_5)

    if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec_5.astype(int).reshape(-1)) % prime

    else:

      tts_results[2]["c"] = (tts_results[2]["c"].to_numpy(dtype=int) + x_vec_5.astype(int).reshape(-1)) % prime

    return tts_results, variance, mean_circular_error


#Idee 6: eine sehr einfache Wahrscheinlichkeitsverteilung für die Abweichungen: mögliche Werte 0, 1, -1, wobei 0 sehr hohe W'keit nahe 1, +-1 gleich wahrscheinlich (W'keit nahe 0)

def get_modulo_data_simple_deviations(prime=101, prob_zero=0.95, validationDataRequired=False, inputError=""):

    prob_one = 0.5*(1-prob_zero)
    prob_minus_one = 0.5*(1-prob_zero)

    tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)

    x_vec_6 = np.random.choice(np.array([0,1,-1]), size=len(tts_results[2]), p = [prob_zero, prob_one, prob_minus_one])

    variance = np.var(x_vec_6)
    mean_circular_error = get_mean_circular_error(prime, x_vec_6)

    if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec_6.astype(int).reshape(-1)) % prime

    else:

      tts_results[2]["c"] = (tts_results[2]["c"].to_numpy(dtype=int) + x_vec_6.astype(int).reshape(-1)) % prime
  
    return tts_results, variance, mean_circular_error



#Datensatz bei Modulo-Addition von 3 Summanden

def get_modulo_data_3_summands(prime=101):

    data_zusatz = pd.DataFrame(columns=["a", "b", "c", "d"])

    k=0
    for a in range(0, prime):
        for b in range(0, prime):
            for c in range (0, prime):
                data_zusatz.loc[k] = [a, b, c, (a+b+c) % prime]
                k = k+1

    return data_zusatz


#Idee 7: Datensatz bei Label Shuffling (Baseline/worst case für Generalisierung)

def get_modulo_data_label_shuffling(prime=101, validationDataRequired=False, inputError=""):


  tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)
  old_labels = list(tts_results[2]["c"])
  new_labels=old_labels.copy()
  random.shuffle(new_labels)


  old_labels = np.array(old_labels)
  new_labels = np.array(new_labels)

  x_vec_7 = np.subtract(new_labels, old_labels)
  variance = np.var(x_vec_7)
  mean_circular_error = get_mean_circular_error(prime, x_vec_7)

  no_error = np.where(x_vec_7 == 0)


  if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec_7.astype(int).reshape(-1)) % prime

  else:

      tts_results[2]["c"] = (tts_results[2]["c"].to_numpy(dtype=int) + x_vec_7.astype(int).reshape(-1)) % prime

  return tts_results, variance, mean_circular_error



#Idee 8: Skellam-Fehlerverteilung

def get_skellam_mod_data(prime=101, lambda1=1, lambda2=1, validationDataRequired=False, inputError=""):

  tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)

  x_vec_8 = scipy.stats.skellam.rvs(lambda1, lambda2, size=len(tts_results[2]))

  variance = np.var(x_vec_8)
  mean_circular_error = get_mean_circular_error(prime, x_vec_8)

  if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec_8.astype(int).reshape(-1)) % prime

  else:

      tts_results[2]["c"] = (tts_results[2]["c"].to_numpy(dtype=int) + x_vec_8.astype(int).reshape(-1)) % prime

  return tts_results, variance, mean_circular_error



#Idee 9: Diskrete von Mises - Verteilung
# wichtig: mu / location Hyperparameter muss radial sein

def get_vonmises_mod_data(prime=101, mu_lin=0, kappa=1, validationDataRequired=False, inputError=""):

  tts_results = get_train_test_split_results(prime, validationDataRequired=validationDataRequired)

  #zuerst diskretisieren --> Argument auch radial
  mu_rad = (mu_lin+int(np.floor(prime/2)))*2*np.pi / prime
  pdf_values=[]
  vm_dist_freeze = scipy.stats.vonmises(loc=mu_rad, kappa=kappa)
  for x in np.arange(0, 2*np.pi, 2*np.pi/prime):
    pdf_values.append(vm_dist_freeze.pdf(x))

  #normalisieren
  pdf_values_norm = np.array(pdf_values) / np.sum(pdf_values)

  x_vec_9 = np.random.choice(range(int(-1*np.floor(prime/2)), int(np.ceil(prime/2))), size=len(tts_results[2]), p=pdf_values_norm)

  variance = np.var(x_vec_9)
  mean_circular_error = get_mean_circular_error(prime, x_vec_9)


  if inputError=="a" or inputError=="b":

      tts_results[0][inputError] = (tts_results[0][inputError].to_numpy(dtype=int) + x_vec_9.astype(int).reshape(-1)) % prime

  else:

      tts_results[2]["c"] = (tts_results[2]["c"].to_numpy(dtype=int) + x_vec_9.astype(int).reshape(-1)) % prime

  return tts_results, variance, mean_circular_error



#Trainingsdurchläufe speichern

def save_training_run(run_name = "", meta_info = None, plots = [], save_dir = "runs"):

    if run_name == "":
        run_name = datetime.datetime.now().isoformat()

    run_path = os.path.join(save_dir, run_name)
    run_path = os.path.join("/kaggle/working/results", run_path)
    os.makedirs(run_path, exist_ok=True)

    with open(os.path.join(run_path, "meta_info.json"), "w") as f:
        json.dump(meta_info, f, indent=2)

    for i, plot in enumerate(plots):
        plot_title = ""
        if i == 0:
            plot_title = "loss.png"
        elif i == 1:
            plot_title = "metrics.png"
        elif i == 2:
            plot_title = "weight_distribution.png"
        elif i == 3:
            plot_title = "dft_plots.png"
        elif i == 4:
            plot_title = "histograms_weights.png"
        elif i == 5:
          plot_title = "pca_plot.png"
        elif i == 6:
          plot_title = "pca_plot_2.png"
        elif i == 7:
          plot_title = "svd_plot_3.png"
        elif i == 8:
          plot_title = "variance_loss_plot.png"
        plot_path = os.path.join(run_path, plot_title)
        plot.savefig(plot_path)

    print(f"Run info saved to directory {run_path} ")



#Prüfen, wo und wie viele Fehler im Datensatz erzeugt wurden

def check_error_impact_on_dataset(prime, X_train, y_train, inputError=""):

  actual_values = y_train["c"]
  expected_values = (np.array(X_train["a"]) + np.array(X_train["b"])) % prime

  if inputError=="a":
    actual_values = np.array(X_train["a"])
    expected_values = (np.array(y_train["c"]) - np.array(X_train["b"])) % prime

  elif inputError=="b":
    actual_values = np.array(X_train["b"])
    expected_values = (np.array(y_train["c"]) - np.array(X_train["a"])) % prime


  error_values = np.array(actual_values) - np.array(expected_values)
  indices = np.where(error_values != 0)[0]
  error_amount = len(indices)
  print(indices)
  print(error_amount)



#Methode, um Punktepaare zu speichern für Varianz - Loss Plot

def variance_loss_plot_save_points(error_type, variance, loss_train, loss_test):

    keys = [
        "no_error", "double_poisson", "uniform", "identic",
        "3_summands", "randomly_dist", "cust_mu_sigma", "simple_deviation",
        "label_shuffle", "skellam", "vonmises", "special"
    ]
    default_dict = {k: [[], []] for k in keys}


    #Prüfen, ob schon das JSON im Drive existiert --> laden und Inhalt aktualisieren
    point_path = "/kaggle/working/results/general/points_variance_loss_plot.json"


    if os.path.exists(point_path):
      print("JSON existiert.")
      with open (point_path, "r") as f:
        points_dict = json.load(f)
      if error_type not in points_dict.keys():
        points_dict[error_type] = [[], []]
    else:
      points_dict = default_dict


    #points_dict = default_dict #wenn man das Punkte-Dict zurücksetzen muss

    points_dict[error_type][0].append((variance, loss_train))
    points_dict[error_type][1].append((variance, loss_test))

    #Speichern
    with open(point_path, "w") as f:
      json.dump(points_dict, f)



In [6]:
#Primzahl erzeugen
prim = generate_prime(100,130)

#Primzahl künstlich setzen
#prim = 107


#Sollen neben Trainings- und Testdaten Validierungsdaten erzeugt werden? (Zur Modellvorhersage später)
validationDataRequired = True

127


In [7]:
#Datensatz normal laden (kein stochastischer Fehler)
def normal_load(er_spec_params):
    if er_spec_params == ['']:
      tf = 0.3
    else:
      tf = er_spec_params[-1]
    dataset = get_train_test_split_results(prim, validationDataRequired = validationDataRequired, train_data_fraction=tf)
    variance = 0.0
    mean_circular_error = 0.0
    return dataset, variance, mean_circular_error

In [8]:
# Datensatz laden --> Hier können verschiedene Funktionen von oben benutzt werden,
#um den Datensatz zu erzeugen inkl. stochastischer Fehler

#Nach Idee 1
def error_idea_1(er_spec_params):
    dataset, variance, mean_circular_error = get_double_poisson_mod_data(prime=prim, lambdaY=er_spec_params[0], lambdaZ=er_spec_params[1], p=0.5, validationDataRequired=validationDataRequired, inputError=er_spec_params[2])
    print(pd.concat([dataset[0], dataset[2]], axis = 1))
    check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
    return dataset, variance, mean_circular_error

In [9]:
#Nach Idee 2
def error_idea_2(er_spec_params):
    dataset, variance, mean_circular_error = get_uniform_mod_data(prime=prim, k=er_spec_params[0], validationDataRequired=validationDataRequired, inputError=er_spec_params[1])
    print(pd.concat([dataset[0], dataset[2]], axis = 1))
    check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
    return dataset, variance, mean_circular_error

In [10]:
#Nach Idee 3
def error_idea_3(er_spec_params):
    dataset, variance, mean_circular_error = get_identic_error_mod_data(prime=prim, validationDataRequired=validationDataRequired, inputError=er_spec_params[0])
    print(pd.concat([dataset[0], dataset[2]], axis = 1))
    check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
    return dataset, variance, mean_circular_error

In [11]:
#Nach Idee 4
def error_idea_4(er_spec_params):
    dataset, variance, mean_circular_error = get_randomly_distributed_error_mod_data(prime=prim, range_of_values=er_spec_params[0], validationDataRequired=validationDataRequired, inputError=er_spec_params[1])
    print(pd.concat([dataset[0], dataset[2]], axis = 1))
    check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
    return dataset, variance, mean_circular_error

In [12]:
#Nach Idee 5
def error_idea_5(er_spec_params):
    dataset, variance, mean_circular_error = get_customized_mu_sigma_modulo_data_no_2(prime=prim, target_mean=er_spec_params[0], target_var=er_spec_params[1], validationDataRequired=validationDataRequired, inputError=er_spec_params[2])
    print(pd.concat([dataset[0], dataset[2]], axis = 1))
    check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
    return dataset, variance, mean_circular_error

In [13]:
#Nach Idee 6
def error_idea_6(er_spec_params):
    dataset, variance, mean_circular_error = get_modulo_data_simple_deviations(prime=prim, prob_zero=er_spec_params[0], validationDataRequired=validationDataRequired, inputError=er_spec_params[1])
    print(pd.concat([dataset[0], dataset[2]], axis = 1))
    check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
    return dataset, variance, mean_circular_error

In [14]:
#Nach Idee 7: Baseline: Label Shuffling: Permutation, also aus Grokking-Sicht schlechtester Fall
def error_idea_7(er_spec_params):
    dataset, variance, mean_circular_error = get_modulo_data_label_shuffling(prime=prim, validationDataRequired=validationDataRequired, inputError=er_spec_params[0])
    print(pd.concat([dataset[0], dataset[2]], axis = 1))
    check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
    return dataset, variance, mean_circular_error

In [15]:
#Nach Idee 8: Skellam-Fehler
def error_idea_8(er_spec_params):
  dataset, variance, mean_circular_error = get_skellam_mod_data(prim, lambda1=er_spec_params[0], lambda2=er_spec_params[1], validationDataRequired=validationDataRequired, inputError=er_spec_params[2])
  print(pd.concat([dataset[0], dataset[2]], axis = 1))
  check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
  return dataset, variance, mean_circular_error

In [16]:
#Nach Idee 9: von Mises Fehler
def error_idea_9(er_spec_params):
  dataset, variance, mean_circular_error = get_vonmises_mod_data(prim, mu_lin=er_spec_params[0], kappa=er_spec_params[1], validationDataRequired=validationDataRequired, inputError=er_spec_params[2])
  print(pd.concat([dataset[0], dataset[2]], axis = 1))
  check_error_impact_on_dataset(prime=prim, X_train=dataset[0], y_train=dataset[2])
  return dataset, variance, mean_circular_error

In [17]:
#Bei Modulo-Addition von 3 Summanden:
def data_3_summands(er_spec_params):
    dataset = get_train_test_split_results(er_spec_params[0], threeSummands=True, validationDataRequired=validationDataRequired, train_data_fraction = er_spec_params[2])
    variance = 0.0
    mean_circular_error = 0.0
    #Nur Trainingsdaten:
    print(pd.concat([dataset[0], dataset[2]], axis = 1))

    return dataset, variance, mean_circular_error

In [18]:
def sweep_create_X_y(dataset):
    X_train, X_test, y_train, y_test = dataset[0], dataset[1], dataset[2], dataset[3]
    X_train = torch.tensor(X_train.values, dtype=torch.float32)
    y_train = torch.tensor(y_train.values, dtype=torch.float32)
    X_test  = torch.tensor(X_test.values,  dtype=torch.float32)
    y_test  = torch.tensor(y_test.values,  dtype=torch.float32)

    #nur für MULT
    '''
        valid_tokens = list(range(1, prim))
        remap = {old: new for new, old in enumerate(valid_tokens)}
        X_train = torch.tensor([[remap[int(v.item())] for v in row] for row in X_train])
        X_test  = torch.tensor([[remap[int(v.item())] for v in row] for row in X_test])
        y_train = torch.tensor([remap[int(v.item())] for v in y_train])
        y_test  = torch.tensor([remap[int(v.item())] for v in y_test])
    '''

    assert X_train.max() < prim - 1, f"X_train max={X_train.max()}"
    assert X_test.max()  < prim - 1, f"X_test max={X_test.max()}"
    assert y_train.max() < prim - 1, f"y_train max={y_train.max()}"
    assert y_test.max()  < prim - 1, f"y_test max={y_test.max()}"
    assert X_train.min() >= 0
    assert y_train.min() >= 0
    print("Alle Checks OK")
    
    X_train = X_train.to(device=device)
    y_train = y_train.to(device=device)
    X_test  = X_test.to(device=device)
    y_test  = y_test.to(device=device)
    return X_train, X_test, y_train, y_test

In [19]:

def sweep_create_mod_data(error_type, error_specific_params):
    if(error_type == "no_error"):
        dataset, variance, mean_circular_error = normal_load(error_specific_params)
    if(error_type == "double_poisson"):
        dataset, variance, mean_circular_error = error_idea_1(error_specific_params)
    if(error_type == "uniform"):
        dataset, variance, mean_circular_error = error_idea_2(error_specific_params)
    if(error_type == "identic"):
        dataset, variance, mean_circular_error = error_idea_3(error_specific_params)
    if(error_type == "randomly_dist"):
        dataset, variance, mean_circular_error = error_idea_4(error_specific_params)
    if(error_type == "cust_mu_sigma"):
        dataset, variance, mean_circular_error = error_idea_5(error_specific_params)
    if(error_type == "simple_deviation"):
        dataset, variance, mean_circular_error = error_idea_6(error_specific_params)
    if(error_type == "3_summands"):
        dataset, variance, mean_circular_error = data_3_summands(error_specific_params)
    if(error_type == "label_shuffle"):
        dataset, variance, mean_circular_error = error_idea_7(error_specific_params)
    if(error_type == "skellam"):
        dataset, variance, mean_circular_error = error_idea_8(error_specific_params)
    if(error_type == "vonmises"):
        dataset, variance, mean_circular_error = error_idea_9(error_specific_params)
    if(error_type == "special"):
        dataset, variance, mean_circular_error = normal_load(error_specific_params)
    return dataset, variance, mean_circular_error

In [20]:
#Funktion zur Erzeugung eines fixen Fourier-Embeddings

def create_fourier_embed(prime=127, d_model=128, K=6):

    d_model_embed = d_model
    if K > d_model/2:
        d_model_embed = 2*K
    
    E = torch.zeros(prime, d_model_embed)
    
    for a in range(0, prime):
        for k in range(1,K+1):
        
            E[a,2*(k-1)] = math.cos(2*math.pi * k * a/ prime)
            E[a, 2*(k-1)+1] = math.sin(2*math.pi * k * a/ prime)

    return E, d_model_embed


In [21]:
#Modell-Definition

class ModModel (nn.Module):

    def __init__(self, d_vocab, d_model, n_ctx, model_type="1_hidden", weight_tying=False):
        super().__init__()
        self.model_type=model_type
        self.W_E = nn.Parameter(torch.randn(d_vocab, d_model) / np.sqrt(d_vocab)) #Embedding-Matrix, wird gelernt --> one-hot-encoding wäre nötig
        #E, d_model_embed = create_fourier_embed(prime=127,d_model=d_model, K=16) #Für Fix Embedding
        #self.W_E = nn.Parameter(E, requires_grad=False) #Für Fix Embedding später
        self.d_1 = n_ctx * d_model #Dimension des konkatenierten Inputs
        self.d_mlp = 4 * n_ctx * d_model #Dimension des MLP
        self.mlp_lin = nn.Linear(self.d_1, self.d_mlp) #MLP
        self.mlp_act = nn.ReLU() #MLP-Aktivierungsfunktion

        if model_type == "1_hidden":
          self.output = nn.Linear(self.d_mlp, d_model, bias=False) #Berechnung des Outputs
        elif model_type == "2_hidden":
          self.d_mlp2 = 2 * n_ctx * d_model
          self.mlp2_lin = nn.Linear(self.d_mlp, self.d_mlp2)
          self.mlp2_act = nn.ReLU()
          self.output = nn.Linear(self.d_mlp2, d_model, bias=False)
        elif model_type == "3_hidden":
          self.d_mlp2 = 2 * n_ctx * d_model
          self.d_mlp3 = n_ctx * d_model
          self.mlp2_lin = nn.Linear(self.d_mlp, self.d_mlp2)
          self.mlp2_act = nn.ReLU()
          self.mlp3_lin = nn.Linear(self.d_mlp2, self.d_mlp3)
          self.mlp3_act = nn.ReLU()
          self.output = nn.Linear(self.d_mlp3, d_model, bias=False)

        
        self.W_U = nn.Parameter(torch.randn(d_model, d_vocab) / np.sqrt(d_model)) #Unembedding-Matrix, wird gelernt

        if weight_tying == True:
          self.output = nn.Linear(self.d_mlp, d_model, bias=False) #Berechnung des Outputs #Für Fix Embedding: d_model_embed
          self.W_U = nn.Parameter(torch.randn(d_model, d_vocab) / np.sqrt(d_model_embed)) #Unembedding-Matrix, wird gelernt Fix Embedding: d_model_embed
          self.W_U.data = self.W_E.data.t()

    def forward(self, x): #x muss Dimension (2 x d_vocab) haben
        x_0 =  self.W_E[x.int(),:] #x @ self.W_E #Embedding --> diese Form von Embedding benötigt einen manuellen one-hot-encoding-Schritt
        #x_0 = self.embed(x.int())
        x_1 = einops.rearrange(x_0, " b p d -> b (p d)") #Konkatenieren
        if self.model_type == "1_hidden":
          x_2 = self.output(self.mlp_act(self.mlp_lin(x_1)))
        elif self.model_type == "2_hidden":
          x_2 = self.output(self.mlp2_act(self.mlp2_lin(self.mlp_act(self.mlp_lin(x_1)))))
        elif self.model_type == "3_hidden":
          x_2 = self.output(self.mlp3_act(self.mlp3_lin(self.mlp2_act(self.mlp2_lin(self.mlp_act(self.mlp_lin(x_1)))))))
        x_3 = x_2 @ self.W_U #Unembedding
        return x_3


In [22]:
#Training-Parameter Standardwerte 
#n_epochs = 30000
#lr = 1e-3
#wd = 1.0
#DATA_SEED = 100
#betas = (0.9, 0.98)
#d_vocab = prim
#d_model = 128
#n_ctx = 2

#Loss-Function Definition (Cross-Entropy-Loss)
def loss_fn(logits, labels):
    if len(logits.shape) == 3:
        print(logits.shape)
        logits = logits[:, -1]
    logits = logits.to(torch.float64)
    log_probs  = logits.log_softmax(dim=-1)
    labels = labels.int()
    correct_log_probs = log_probs.gather(dim=-1, index=labels.unsqueeze(1)).squeeze(1)
    #correct_log_probs = log_probs.gather(dim=-1, index=labels)[:, 0]
    return -correct_log_probs.mean()


#Funktion zur Umsetzung von Weight Noise (Relativ oder Absolut)
def weight_noise_fn(model, rel_or_absolute_std = "absolute", std_value = 0.01):

  with torch.no_grad():
    for parameter in model.parameters():
      if parameter.requires_grad:
        noise = torch.randn_like(parameter)
        if rel_or_absolute_std == "relative":
          parameter.add_(parameter * std_value * noise)
        elif rel_or_absolute_std == "absolute":
          parameter.add_(std_value * noise)



#Funktion zur Betimmung von Grokking-Start und Grokking-Ende
def determine_grokking_time(test_losses):

  grokking_start_and_end = []

  for epoch, tl in enumerate(test_losses):

        if epoch % 100 == 0 and epoch >= 200:
          if (test_losses[epoch-100]/test_losses[epoch-200]) > 0.9 and tl/test_losses[epoch-100] < 0.9:
              grokking_start_and_end.append(epoch)
          if (test_losses[epoch-100]/test_losses[epoch-200]) < 0.9 and tl/test_losses[epoch-100] > 0.9:
              grokking_start_and_end.append(epoch)

  if grokking_start_and_end == []:
    return "no grokking"

  return grokking_start_and_end


#Funktion zur Nutzung von Logit Noise
def logit_noise_fn(train_logits, std = 0.001, apply=False, mode = "absolute"):
    if not apply:
        return train_logits
        
    error_vec1 = torch.randn_like(train_logits)
    
    if mode == "absolute":
        error_vec2 = error_vec1 * std
    elif mode == "relative":
        error_vec2 = error_vec1 * std * train_logits.abs() #Größere Werte bestrafen
    
    train_logits_noisy = train_logits + error_vec2
    return train_logits_noisy


In [23]:
train_losses=[]
test_losses=[]
metrics=[]
weights=[]
weights_for_hists = []
initial_test_loss = 0.0
final_train_loss = 0.0
final_test_loss = 0.0

def set_train_params_back():
    train_losses=[]
    test_losses=[]
    metrics=[]
    weights=[]
    weights_for_hists = []
    initial_test_loss = 0.0
    final_train_loss = 0.0
    final_test_loss = 0.0

    return train_losses, test_losses, metrics, weights, weights_for_hists, initial_test_loss, final_train_loss, final_test_loss

def sweep_training(X_train, X_test, y_train, y_test, d_vocab, d_model, context_len, model_type, weight_tying, learning_rate, betas, weight_decay, weight_noise):

    #Modell und Optimierer initialsieren
    print(f"d_vocab={d_vocab}, X_train max={X_train.max().item()}, X_test max={X_test.max().item()}")
    model = ModModel(d_vocab, d_model, context_len, model_type, weight_tying).to(device=device)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=betas, weight_decay=weight_decay)

    #Training
    for epoch in tqdm(range(n_epochs)):
        
        if weight_noise:
          weight_noise_fn(model, rel_or_absolute_std="absolute", std_value=0.001)
        
        train_logits = model(X_train)
        #train_logits = logit_noise_fn(train_logits, std = 10.0, apply = True, mode = "absolute") ### Nur für LOGIT NOISE
        train_loss = loss_fn(train_logits, y_train)
        train_loss.backward()
        train_losses.append(train_loss.item())
        optimizer.step()
        optimizer.zero_grad()
        with torch.inference_mode():
            test_logits = model(X_test)
            test_loss = loss_fn(test_logits, y_test)
            test_losses.append(test_loss.item())
            metrics.append(get_metrics(test_logits, y_test, epoch))
            W_output=model.output.weight.detach().cpu().numpy()

            weights.append(   [np.mean(np.absolute(W_output) > 1.0),
                              np.mean(np.absolute(W_output) <= 1.0),
                              np.mean(np.absolute(W_output) <= 0.1),
                              np.mean(np.absolute(W_output) <= 0.01),
                              np.mean(np.absolute(W_output)<= 0.001),
                              np.mean(np.absolute(W_output) <= 0.0)])

            if (epoch % 10000 == 0):
              weights_for_hists.append(W_output)

            elif (epoch == n_epochs-1):
              weights_for_hists.append(W_output)

        if (epoch == 0):
            initial_test_loss = test_loss.item()

        if (((epoch+1)%5000) == 0):
            print(f"Train loss: {train_loss.item()}   Test loss: {test_loss.item()}")

    f_train_loss = train_loss.item()
    f_test_loss = test_loss.item()

    #Cleanup
    del optimizer
    torch.cuda.empty_cache()
    return model, f_train_loss, f_test_loss, initial_test_loss


In [24]:
#Methode, um zu entscheiden, ob man von Grokking sprechen kann
#Definiere: Grokking liegt vor, wenn Test-Loss mind. 100 mal kleiner zu Beginn als am Ende des Trainings

def check_for_grokking(initial_test_loss, final_test_loss):
    grokking_observed = False
    if (initial_test_loss >= 100 * final_test_loss):
        grokking_observed = True
    return grokking_observed

#Methode, um Modell auf 3 Summanden Modulo-Addition umzustellen:
def sweep_get_context_length(current_error_type):
    if current_error_type == "3_summands":
        return 3
    else:
        return 2


In [25]:
#Auswertung durch plots
def sweep_loss_plot():
    #Loss Plot
    loss_fig, ax1 = plt.subplots()
    ax1.plot(range(n_epochs), train_losses[:n_epochs:], label="Training Loss")
    ax1.plot(range(n_epochs), test_losses[:n_epochs:], label="Test Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Log Loss")
    ax1.set_yscale("log")
    ax1.set_title("Training and Test loss")
    ax1.legend()
    return loss_fig

#Plot der Metriken Accuracy, Precision, Recall und F1-Score
def sweep_metrics_plot():
    metrics_lst = list(map(list, zip(*metrics)))
    metrics_fig, ((ax11, ax2), (ax3, ax4)) = plt.subplots(2,2)
    ax11.plot(range(n_epochs), metrics_lst[0], color="r", label="Accuracy"); ax11.set_title("Accuracy")
    ax2.plot(range(n_epochs), metrics_lst[1], color="b", label="Precision"); ax2.set_title("Precision")
    ax3.plot(range(n_epochs), metrics_lst[2], color="c", label="Recall"); ax3.set_title("Recall")
    ax4.plot(range(n_epochs), metrics_lst[3], color="g", label="F1-Score"); ax4.set_title("F1-Score")
    for ax in (ax11, ax2, ax3, ax4):
        ax.set_xlabel("Epoch")
        ax.legend

    metrics_fig.suptitle("Metrics")
    metrics_fig.tight_layout()
    return metrics_fig


In [26]:
#Plot der Verteilung der Gewichte im Final Layer des MLP

def sweep_weight_dist_plot():

    W = np.array(weights)

    weights_greater_1 = W[:, 0]
    weights_sub_1 = W[:, 1]
    weights_sub_0_1 = W[:, 2]
    weights_sub_0_01 = W[:, 3]
    weights_sub_0_001 = W[:, 4]
    weights_less_0 = W[:, 5]


    weight_dist_fig, ax = plt.subplots(figsize=(15,8))
    ax.plot(range(n_epochs), 100*weights_less_0, linestyle="--", color="r", label="|w| <= 0")
    ax.plot(range(n_epochs), 100*weights_sub_0_001, linestyle="--", color="b", label="|w| <= 0.001")
    ax.plot(range(n_epochs), 100*weights_sub_0_01, linestyle="--", color="g", label="|w| <= 0.01")
    ax.plot(range(n_epochs), 100*weights_sub_0_1, linestyle="--", color="c", label="|w| <= 0.1")
    ax.plot(range(n_epochs), 100*weights_sub_1, linestyle="--", color="m", label="|w| <= 1")
    ax.plot(range(n_epochs), 100*weights_greater_1, linestyle="--", color="y", label="|w| > 1")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("% of weights")
    ax.legend(loc="center right", bbox_to_anchor=(1.2,0.5))
    weight_dist_fig.suptitle("Weight distribution")
    weight_dist_fig.tight_layout()
    return weight_dist_fig


#Plot der Histogramme der Gewichte zu verschiedenen Trainingszeitpunkten
def sweep_hist_weights_plot():
    for i in range(len(weights_for_hists)):
      print(weights_for_hists[i].shape)

    amount_rows = int(len(weights_for_hists)/2)
    if len(weights_for_hists) % 2 == 1:
      amount_rows = int((len(weights_for_hists) +1)/2)

    histograms_weights_fig, hist_axs = plt.subplots(amount_rows, 2, figsize=(8, 8))

    for i, ax in enumerate(hist_axs.flatten()):
      if i < len(weights_for_hists):
        ax.hist(weights_for_hists[i].flatten(), bins = 50, density = True, edgecolor = "black")
        sns.kdeplot(weights_for_hists[i].flatten(), ax=ax, color="red")
        ax.set_title(f"Hist. weight dist. Epoch {i*10}k")
        ax.set_xlabel("weight size")

    histograms_weights_fig.suptitle("Weight distribution: Histograms for selected epochs")
    histograms_weights_fig.tight_layout()
    return histograms_weights_fig

In [27]:
#Validierung: Modell-Vorhersagen für bisher unberücksichtigte Validierungsdaten
#falls Validierungsdaten nicht vorhanden: betrachte gesamten Datensatz

amount_preds_wrong = 0.0
amount_preds_total = 0.0
share_preds_wrong = 0.0

def set_validation_params_back():
    amount_preds_wrong = 0.0
    amount_preds_total = 0.0
    share_preds_wrong = 0.0
    return amount_preds_wrong, amount_preds_total, share_preds_wrong

def sweep_validate(dataset, model):
    if validationDataRequired:
      X_val = dataset[4]
      y_val = dataset[5]

    else:
      data2, variance = normal_load()
      X_val = data2[["a", "b"]]
      y_val = data2[["c"]]

  

    X_2 = torch.tensor(X_val.values, dtype=torch.float32)
    y_2 = torch.tensor(y_val.values, dtype=torch.float32)

    #Nur für Modulo-Multiplikation
    '''
        valid_tokens = list(range(1, prim))
        remap = {old: new for new, old in enumerate(valid_tokens)}
        X_2 = torch.tensor([[remap[int(v.item())] for v in row] for row in X_2])
        y_2 = torch.tensor([remap[int(v.item())] for v in y_2.squeeze()])
    '''
    X_2 = X_2.to(device=device)
    y_2 = y_2.to(device=device)
    
    model.eval()

    with torch.inference_mode():
      logits_2 = model(X_2)

    amount_preds_wrong, amount_preds_total = get_metrics(logits_2, y_2, 0, "pred")
    share_preds_wrong = round(amount_preds_wrong / amount_preds_total, 4)

In [28]:
#Reverse Engineering
def sweep_ft_plot(model):

        ft_fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8)) = plt.subplots(4,2, figsize=(30,45))
        ft_fig.suptitle("DFTs")
        plt.subplots_adjust(left=0.1, bottom=0.1, hspace=0.2, wspace=0.2)

        embedding_matrix = model.W_E.detach().cpu().numpy()
        unembedding_matrix = model.W_U.detach().cpu().numpy()
        mlp_output_matrix = model.output.weight.detach().cpu().numpy()
        W_L = unembedding_matrix.T @ mlp_output_matrix


        #Modulo-Mult: Sparse Spektrum erzeugen
        '''
            log_order_ = []
            log_indices_0_based = [x - 1 for x in log_order_107]
            embedding_matrix = embedding_matrix[log_indices_0_based]
        '''

        embed_dft = scipy.fft.fft2(embedding_matrix)

        #Untersuche jetzt nicht die 2D DFT, sondern einzelne Dimensionen

        #siehe Plot: man erkennt: wenn man für jede Spalte eine DFT durchführt und danach für jede Zeile die L2 - Norm bestimmt,
        #dass für gewisse Zeilenindizes (die jeweils für eine bestimmte Frequenz stehen) hohe Peaks bestehen. Das bedeutet, dass diese
        #wenigen Basis-Frequenzen genügen, um die Embeddings sehr gut zu erklären bzw. zu repräsentieren.

        embed_dft_each_column = scipy.fft.fft(embedding_matrix, axis=0)

        norm_over_rows = np.linalg.norm(embed_dft_each_column, axis=1)
        key_frequencies = np.where(norm_over_rows > 10)
        print(f"Schlüsselfrequenzen: {key_frequencies}")
        ax1.plot(norm_over_rows)
        ax1.set_xlabel("d_vocab / Frequency k")
        ax1.set_ylabel("L2-Norm")
        ax1.set_title("L2-Norm der Zeilen der FT-Embedding-Matrix")


        #Betrachte jetzt die Neuron-Logit Map W_L = W_U.T @ W_output, Dimension (P=109, 1024)
        wl_dft_2 = scipy.fft.fft2(W_L)
        wl_dft_2 = np.absolute(wl_dft_2)
        wl_dft_2 =  wl_dft_2 / np.max(wl_dft_2)

        wl_dft_1 = scipy.fft.fft(W_L, axis=0)
        norm_wl_dft_1 = np.linalg.norm(wl_dft_1, axis=1)

        ax2.plot(norm_wl_dft_1)
        ax2.set_xlabel("Frequency k")
        ax2.set_ylabel("L2-Norm über die Neuron-Achse")
        ax2.set_title("L2-Norm der Zeilen der Neuron-Logit-Map W_L")

        ax4.imshow(wl_dft_2, cmap="grey", aspect="auto")
        ax4.set_xlabel("Dimensionen / Spalten")
        ax4.set_ylabel("Logits / Frequenz k")
        ax4.set_title("Aktivierte Frequenzen der FT-Neuron-Logit-Matrix")

        x = np.absolute(embed_dft)
        max_x = np.max(x)
        x = x/max_x
        ax3.imshow(x,cmap="grey", aspect="auto")
        ax3.set_xlabel("Interne Modelldimension")
        ax3.set_ylabel("Frequenz k")
        ax3.set_title("Aktivierte Frequenzen der FT-Embedding-Matrix")

        #Betrachte zuletzt noch eine geeignete Spalte: vor DFT: Sinus/Cosinus-Form (möglichst klar erkennbar mit einer Frequenz), nach DFT: Energie des Signals möglichst konzentriert an einer Frequenz

        #Schritt 1: berechne für jede Spalte sparsity scores, die messen, wie sehr sich eine Spalte an einer Frequenz konzentriert

        scores = []
        for column in embed_dft_each_column.T:
          y = np.square(np.abs(column))
          sparsity_score = max(y) / np.sum(y)
          scores.append(sparsity_score)

        #Schritt 2: Hole die Spalte, die den maximalen Score erzielt:
        max_score_column_pre_dft = embedding_matrix[:, np.where(scores == max(scores))[0]]
        max_score_column_post_dft = embed_dft_each_column[:, np.where(scores == max(scores))[0]]

        only_key_freqs = np.array([value[0] if np.isin(key, key_frequencies) else 0.0 for key, value in enumerate(max_score_column_post_dft)])
        only_key_freqs_ifft = scipy.fft.ifft(only_key_freqs)


        #Schritt 3: plotte diese Spalte gegen 0...P-1 (sowohl pre als auch post DFT)
        ax5.plot(range(prim), max_score_column_pre_dft, linestyle="--", marker="o")
        ax5.set_xlabel("Werte 0...P-1")
        ax5.set_ylabel("Signalwert")
        ax5.set_title("Spalte/Signal mit maximalem Sparsity-Score")

        ax6.plot(range(prim),np.abs(max_score_column_post_dft), linestyle="--", marker="o")
        ax6.set_xlabel("Frequenz k")
        ax6.set_ylabel("Signalwert für Frequenz k (Frequenzbereich)")
        ax6.set_title("Spalte/Signal-Amplitude mit maximalem Sparsity-Score nach DFT")

        ax7.plot(range(prim), only_key_freqs_ifft, linestyle="-", marker="o", color="r", label="Inverse FT der key freqs")
        ax7.set_xlabel("WErte 0...P-1")
        ax7.set_ylabel("Signalwert der inversen FT")
        ax7.set_title("Inverse FT nur mit den Hauptfrequenzen")

        #Plotten der Komplexen Ebene einer Zeile: wie lernt das Modell die Phase?
        if key_frequencies[0].size > 0:
            zeile = np.random.choice(key_frequencies[0])
            row_signal_post_dft = embed_dft_each_column[zeile, :] / np.absolute(embed_dft_each_column[zeile, :])
            points_complex_plane_real = [k.real for k in row_signal_post_dft]
            points_complex_plane_imag = [k.imag for k in row_signal_post_dft]

            ax8.scatter(points_complex_plane_real, points_complex_plane_imag,  color="black")
            ax8.set_ylabel("Im(c_k)")
            ax8.set_xlabel("Re(c_k)")
            ax8.set_title(f"alle Fourierkomponenten für Zeile/Frequenz {zeile}")
        else:
            ax8.set_title("No key frequencies found for complex plane plot")
            ax8.text(0.5, 0.5, 'No key frequencies found', transform=ax8.transAxes,
                     horizontalalignment='center', verticalalignment='center')

        ft_fig.tight_layout()
        return ft_fig


In [29]:
#PCA-Plots der Embedding-Matrix

def sweep_pca_svd_plot(model):

  embedding_matrix = model.W_E.detach().cpu().numpy()
  unembedding_matrix = model.W_U.detach().cpu().numpy()

  #Embedding-Matrix hat Dimension d_vocab x d_model, Unembedding d_model x d_vocab
 
  E = embedding_matrix.astype(np.float64)
  U = unembedding_matrix.astype(np.float64)
  if E.shape[1] != U.shape[0]:
      L = E[:, :128]@U
  else:
      L=E@U

  #Zentrieren
  E_centered = E - np.mean(E, axis=0)
  U_centered = U - np.mean(U, axis=0)
  L_centered = L - np.mean(L, axis=0)


  #Weg 1: über die Kovarianzmatrix
  cov_matrix_e = np.cov(E_centered, rowvar=False)
  cov_matrix_u = np.cov(U_centered, rowvar=True)
  cov_matrix_l = np.cov(L_centered, rowvar=False)

  def pca_weight_matrices(C):
    eigvals, eigvecs = np.linalg.eigh(C)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    return eigvals, eigvecs

  eigenvalues_e, eigenvectors_e = pca_weight_matrices(cov_matrix_e)
  Z_e = E_centered @ eigenvectors_e

  eigenvalues_u, eigenvectors_u = pca_weight_matrices(cov_matrix_u)
  Z_u = U_centered.T @ eigenvectors_u

  eigenvalues_l, eigenvectors_l = pca_weight_matrices(cov_matrix_l)
  Z_l = L_centered @ eigenvectors_l

  #erste drei Komponenten
  first_pc_e = Z_e[:,0]
  second_pc_e = Z_e[:,1]
  third_pc_e = Z_e[:,2]
  first3_pc_e = Z_e[:, 0:2].T

  first_pc_u = Z_u[:,0]
  second_pc_u = Z_u[:,1]
  third_pc_u = Z_u[:,2]
  first3_pc_u = Z_u[:, 0:2].T

  first_pc_l = Z_l[:, 0]
  second_pc_l = Z_l[:, 1]
  third_pc_l = Z_l[:, 2]
  first3_pc_l = Z_l[:, 0:2].T

  #PCA Plot erzeugen

  pca_fig = plt.figure()
  ax1 = pca_fig.add_subplot(2, 2, 1)
  ax2 = pca_fig.add_subplot(2, 2, 2)
  ax3 = pca_fig.add_subplot(2, 2, 3)
  ax4 = pca_fig.add_subplot(2, 2, 4, projection='3d')

  pca_fig.set_size_inches(20,10)
  ax1.scatter(first_pc_e, second_pc_e, c="b")
  ax2.scatter(first_pc_u, second_pc_u, c="r")
  ax3.scatter(first_pc_l, second_pc_l, c="g")


  for i, txt in enumerate(range(prim)):
    ax1.annotate(txt, (first_pc_e[i], second_pc_e[i]))
    if i==1:
      print(f"Für 1: Wert der ersten zwei PC: ({first_pc_e[1], second_pc_e[1]})")
    ax2.annotate(txt, (first_pc_u[i], second_pc_u[i])),
    ax3.annotate(txt, (first_pc_l[i], second_pc_l[i]))


  ax1.set_xlabel("PC1")
  ax1.set_ylabel("PC2")
  ax2.set_xlabel("PC1")
  ax2.set_ylabel("PC2")
  ax3.set_xlabel("PC1")
  ax3.set_ylabel("PC2")
  ax1.set_title("PCA: Embedding-Matrix")
  ax2.set_title("PCA: Unembedding-Matrix")
  ax3.set_title("PCA: Input-Space-to-Logit-Space")


  #hier noch einen 3-dimensionalen Plot von drei Hauptkomponenten

  pc_index = np.random.randint(0, Z_e.shape[1]-2)
  ax4.scatter(Z_e[:,pc_index], Z_e[:,pc_index+1], Z_e[:,pc_index+2], c="b")
  ax4.set_xlabel(f"PC{pc_index+1}")
  ax4.set_ylabel(f"PC{pc_index+2}")
  ax4.set_zlabel(f"PC{pc_index+3}")
  ax4.set_title(f"PCA: Embedding-Matrix (PC #{pc_index+1}, #{pc_index+2}, #{pc_index+3})")


  #Plot aller PCA-Komponenten betrachten

  pca_fig2, axes = plt.subplots(int(np.ceil(Z_e.shape[1]/5)), 5)
  pca_fig2.set_size_inches(40,120)

  for i, ax in enumerate(pca_fig2.axes):

    ax.set_xlabel(f"PC{i+1}")
    ax.set_ylabel(f"PC{i+2}")
    ax.set_title(f"PCA: Embedding-Matrix")

    if i == Z_e.shape[1]-1:
        ax.scatter(Z_e[:,i], Z_e[:,0], c="b")
        ax.set_ylabel(f"PC{0}")
    elif i>=Z_e.shape[1]:
      continue
    else:
      ax.scatter(Z_e[:, i], Z_e[:,i+1], c="b")


  pca_fig2.tight_layout()


  #Plot-Idee 2: Über die SVD-Methode

  U_e, S_e, V_e_t = np.linalg.svd(E_centered, full_matrices=False)
  print(f"Shape der V Matrix bei der SVD: {V_e_t.shape}")

  Z_e_SVD = V_e_t @ E_centered.T
  print(f"Shape der Z Matrix bei der SVD: {Z_e_SVD.shape}")

  svd_fig, ax1 = plt.subplots()
  x=Z_e_SVD[0, :]
  y=Z_e_SVD[1, :]
  ax1.scatter(x, y, c="b")

  for i, txt in enumerate(range(prim)):
    ax1.annotate(txt, (x[i], y[i]))

  ax1.set_xlabel("dim 1")
  ax1.set_ylabel("dim 2")

  ax1.set_title("SVD: Embedding-Matrix")
  svd_fig.tight_layout()

  return pca_fig, pca_fig2, svd_fig

In [30]:
#Trainingsdurchlauf speichern

def sweep_save(plots, current_params, current_error_specific_params, variance, mean_circular_error, final_train_loss, final_test_loss, initial_test_loss, additional_remarks="-"):

    run_name = datetime.datetime.now().isoformat()
    meta_info = {
        "run-name/timestamp": run_name,
        "data": "modulo_addition",
        "random_seed": current_params[0],
        "error_method": current_params[1],
        "other_params": current_error_specific_params,
        "label_or_input_error": current_error_specific_params[-1],
        "error_variance": variance,
        "mean_circular_error": mean_circular_error,
        "final_train_loss": final_train_loss,
        "final_test_loss": final_test_loss,
        "epochs": n_epochs,
        "prime": prim,
        "predictions_total": amount_preds_total,
        "predictions_wrong": amount_preds_wrong,
        "share_predictions_wrong": share_preds_wrong,
        "grokking_observed": check_for_grokking(initial_test_loss, final_test_loss),
        "grokking_start_and_end": determine_grokking_time(test_losses),
        "additional_remarks": additional_remarks
    }

    save_dir = "runs"
    variance_loss_plot_save_points(error_type=current_error_type, variance=variance, loss_train=final_train_loss, loss_test=final_test_loss)

    var_loss_fig = sweep_var_loss_plot(current_error_type)
    plots.append(var_loss_fig)

    save_training_run(run_name=run_name, meta_info=meta_info, plots=plots, save_dir=save_dir)

    plt.close("all")

In [31]:

#Jetzt aktuelles JSON mit den Punkten holen

def sweep_var_loss_plot(current_error_type):

    filepath = "/kaggle/working/results/general/points_variance_loss_plot.json"

    if os.path.exists(filepath):
        with open(filepath, "r") as f:
          plot_points_dict =json.load(f)

    error_types = plot_points_dict.keys()
    key = current_error_type

    points_var_trainloss = plot_points_dict[key][0]
    points_var_testloss = plot_points_dict[key][1]
    x_trainloss = [point[0] for point in points_var_trainloss]
    y_trainloss = [point[1] for point in points_var_trainloss]
    x_testloss = [point[0] for point in points_var_testloss]
    y_testloss = [point[1] for point in points_var_testloss]

    var_loss_fig, ax = plt.subplots()
    ax.scatter(x_trainloss, y_trainloss, c="b", label="Train-Loss")
    ax.scatter(x_testloss, y_testloss, c="orange", label="Test-Loss")
    ax.set_xlabel("Variance")
    ax.set_ylabel("Loss")
    ax.set_yscale("log")
    ax.legend(loc="center right", bbox_to_anchor = (1.2, 0.5))
    var_loss_fig.suptitle(f"Variance-Loss-Plot (with error type: {key})")
    var_loss_fig.tight_layout()
    return var_loss_fig

In [32]:
def sweep_create_plots(model, current_error_type):
    loss_fig = sweep_loss_plot()
    metrics_fig = sweep_metrics_plot()
    weight_dist_fig = sweep_weight_dist_plot()
    histograms_weights_fig = sweep_hist_weights_plot()
    ft_fig = sweep_ft_plot(model)
    pca_fig, pca_fig2, svd_fig = sweep_pca_svd_plot(model)
    #var_loss_fig = sweep_var_loss_plot(current_error_type)
    plots = [loss_fig, metrics_fig, weight_dist_fig, ft_fig, histograms_weights_fig, pca_fig, pca_fig2, svd_fig] #, var_loss_fig]
    return plots

In [33]:
def sweep_special_param_setting(current_error_type, current_error_specific_params):

  if current_error_type == "special":
    n_epochs = current_error_specific_params[0]
    model_type = current_error_specific_params[1]
    weight_tying = current_error_specific_params[2]
    d_model = current_error_specific_params[3]
    lr = current_error_specific_params[4]
    weight_decay = current_error_specific_params[5]
    weight_noise=current_error_specific_params[6]
    train_data_fraction = current_error_specific_params[7]

  else:
    n_epochs = 30000
    model_type = "1_hidden"
    weight_tying = False
    d_model = 128
    lr = 1e-3
    weight_decay = 1.0
    weight_noise=False
    train_data_fraction = 0.3

  betas = (0.9, 0.98)
  d_vocab = prim - 1 # nur für Mult!!!

  return n_epochs, model_type, weight_tying, d_model, lr, weight_decay, weight_noise, train_data_fraction, betas, d_vocab

In [34]:
def sweep_get_last_parameter_index():

    filepath = "/kaggle/working/support"
    os.makedirs(filepath, exist_ok = True)
    with open(os.path.join(filepath, "parameterIndex.txt") , "r") as file:
        last_index = file.read()
        if last_index != "":
            last_index = int(last_index)
        else:
            last_index = -1


    print(f"Last index {last_index}")
    return last_index


def sweep_save_last_parameter_index(i):

    os.makedirs("/kaggle/working/support", exist_ok = True)
    with open("/kaggle/working/support/parameterIndex.txt", "w") as file:
        file.write(str(i))
    

def sweep_is_forbidden_index(i):
    with open("/kaggle/working/support/forbiddenIndices.txt", "r") as file:
        content = file.read().strip().rstrip(",")
        liste = [int(x) for x in content.split(",") if x]
        print(liste)
        if i in liste:
            return True
    return False

def sweep_save_forbidden_index(i):
    with open("/kaggle/working/support/forbiddenIndices.txt", "a") as file:
        file.write(f"{i},")
    

In [ ]:



#Sweep-Durchführung: hier jetzt Iteration der Trainingsdurchläufe

all_parameter_combinations = parameter_sweep() #hier optional den Error-Typ angeben in der Klammer
n_runs1 = len(all_parameter_combinations) - sweep_get_last_parameter_index()
print(f"Ausstehende Runs {n_runs1}")
n_runs2 = 10
count = 0
preferred_runs= []


for j in range(n_runs1):

    #Um maximale Run-Anzahl zu kontrollieren
    count=count+1
    if count > n_runs2:
        break

    print("\n********************")
    print("TRAINING START\n")

    #Aktuellsten Index erhalten
    #i = sweep_get_last_parameter_index() + 1

    #Falls eine bestimmte Parameter-Auswahl trainiert werden soll:
    if preferred_runs != []:
        pref_index = 0 #random.randint(0, len(preferred_runs)-1)
        current_params = preferred_runs[pref_index]
        preferred_runs.pop(pref_index)
    else:
        i = np.random.randint(len(all_parameter_combinations)-1)
        #Index-Manipulation:
        if 0<=i<=20000:
            i = np.random.randint(0, 20000)
        #Aktuelle Parameter/Fehlertyp
        current_params = all_parameter_combinations[i]
        print(f"aktueller Index: {i}")

    print(f"aktuelle Parameter: {current_params}")


    current_random_seed = current_params[0]
    np.random.seed(current_random_seed)

    current_error_type = current_params[1]

    #Um nur bestimmte Fehlertypen laufen zu lassen:
    if ("i" in locals() or "i" in globals()):
        if sweep_is_forbidden_index(i):
            continue
    if (current_error_type not in []): # "uniform", "no_error", "simple_deviation", "identic, "double_poisson", "skellam", "vonmises", "3_summands", "label_shuffle", "cust_mu_sigma", "randomly_dist"
        #sweep_save_last_parameter_index(i)
        continue

    current_error_specific_params = []
    for q in range(2, len(current_params)):
        current_error_specific_params.append(current_params[q])
        
    if current_error_type == "no_error" or current_error_type == "3_summands":
        current_error_specific_params.append('')

    #Daten erzeugen
    dataset, variance, mean_circular_error = sweep_create_mod_data(current_error_type, current_error_specific_params)

    #Daten für Training vorbereiten
    X_train, X_test, y_train, y_test = sweep_create_X_y(dataset)

    #Modell
    context_len = sweep_get_context_length(current_error_type)

    n_epochs, model_type, weight_tying, d_model, learning_rate, weight_decay, weight_noise, train_data_fraction, betas, d_vocab = sweep_special_param_setting(current_error_type, current_error_specific_params)

    #Manipulation einzelner Hyperparameter
    # n_epochs, model_type, weight_tying, d_model, learning_rate, weight_decay,  train_data_fraction, betas, d_vocab
    additional_remarks = "-"

    
    if current_error_type == "3_summands":
      weight_decay = current_error_specific_params[1]

    model, final_train_loss, final_test_loss, initial_test_loss = sweep_training(X_train, X_test, y_train, y_test, d_vocab, d_model, context_len, model_type, weight_tying, learning_rate, betas, weight_decay, weight_noise)

    #Validierung
    sweep_validate(dataset,model)

    #Speichern
    sweep_save(plots=sweep_create_plots(model, current_error_type), current_params=current_params, current_error_specific_params=current_error_specific_params, variance=variance, mean_circular_error=mean_circular_error, final_train_loss=final_train_loss, final_test_loss=final_test_loss, initial_test_loss=initial_test_loss, additional_remarks=additional_remarks)

    #Trainingsparameter und Validierungsparameter zurücksetzen:
    train_losses, test_losses, metrics, weights, weights_for_hists, initial_test_loss, final_train_loss, final_test_loss = set_train_params_back()
    amount_preds_wrong, amount_preds_total, share_preds_wrong = set_validation_params_back()

    #Index sichern
    #sweep_save_last_parameter_index(i)
    if preferred_runs == []:
        sweep_save_forbidden_index(i)

    print("\nTRAINING END")
    print("********************\n")